## Import Libraries

In [143]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.ensemble import RandomForestRegressor, GradientBoostingClassifier
from xgboost import XGBRegressor
from catboost import CatBoostRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler
import numpy as np
import pandas as pd

## Read CSV

In [144]:
df = pd.read_csv("zameen_data(2).csv")

In [145]:
print(f"Dataset shape: {df.shape}")
print(df.head())

Dataset shape: (323, 10)
  property_type  servant_quarters  floors  parking_spaces furnished  \
0         House                 1       2               1        No   
1         House                 0       1               4       Yes   
2         House                 2       3               6       Yes   
3         House                 0       1               1        No   
4         House                 2       2               0        No   

   Price_PKR  Area_SqFt  Beds_Num  Baths_Num  Built_Year_Clean  
0    8500000      652.5         2          2              2024  
1    9800000      900.0         2          3              2023  
2    9900000     1125.0         3          3              2015  
3    9950000     1260.0         3          3              2025  
4   10000000     1125.0         2          3              1980  


## Remove Duplicated Records

In [146]:
print(f"Duplicates before: {df.duplicated().sum()}")

df = df.drop_duplicates()

print(f"Duplicates after: {df.duplicated().sum()}")
print(f"Dataset shape after: {df.shape}")


Duplicates before: 27
Duplicates after: 0
Dataset shape after: (296, 10)


## Missing Values

In [147]:
# As I have already removed the missing values in the data cleaning, it is no longer required

print("Missing values per column:")
print(df.isnull().sum())

Missing values per column:
property_type       0
servant_quarters    0
floors              0
parking_spaces      0
furnished           0
Price_PKR           0
Area_SqFt           0
Beds_Num            0
Baths_Num           0
Built_Year_Clean    0
dtype: int64


## Convert Categorical Variables to Numerical Form

In [148]:
label_encoder = LabelEncoder()

df['furnished_encoded'] = label_encoder.fit_transform(df['furnished'])

df['property_type_encoded'] = label_encoder.fit_transform(df['property_type'])

df = df.drop(columns=['furnished', 'property_type'])
print("\nColumns after encoding:")
print(df.columns.tolist())
print(df.head())



Columns after encoding:
['servant_quarters', 'floors', 'parking_spaces', 'Price_PKR', 'Area_SqFt', 'Beds_Num', 'Baths_Num', 'Built_Year_Clean', 'furnished_encoded', 'property_type_encoded']
   servant_quarters  floors  parking_spaces  Price_PKR  Area_SqFt  Beds_Num  \
0                 1       2               1    8500000      652.5         2   
1                 0       1               4    9800000      900.0         2   
2                 2       3               6    9900000     1125.0         3   
3                 0       1               1    9950000     1260.0         3   
4                 2       2               0   10000000     1125.0         2   

   Baths_Num  Built_Year_Clean  furnished_encoded  property_type_encoded  
0          2              2024                  0                      1  
1          3              2023                  1                      1  
2          3              2015                  1                      1  
3          3              2025    

## Train-test Spilt

In [149]:
X = df.drop(columns=['Price_PKR'])
y = df['Price_PKR']

# Split the data (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split( X, y, test_size=0.2, random_state=42,shuffle=True)

print(f"\nTraining set size: {X_train.shape}")
print(f"Test set size: {X_test.shape}")
print(f"\nTraining price range: {y_train.min():,} to {y_train.max():,}")
print(f"Test price range: {y_test.min():,} to {y_test.max():,}")


Training set size: (236, 9)
Test set size: (60, 9)

Training price range: 8,500,000 to 88,000,000
Test price range: 10,000,000 to 70,000,000


## Scaling features

In [150]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

results = []


## Feature Enigneerning

In [151]:
df['total_rooms'] = df['Beds_Num'] + df['Baths_Num']
df['rooms_per_floor'] = df['total_rooms'] / df['floors'].replace(0, 1)
df['bed_bath_ratio'] = df['Beds_Num'] / df['Baths_Num'].replace(0, 1)

# 2. Area-based features
df['area_per_bedroom'] = df['Area_SqFt'] / df['Beds_Num'].replace(0, 1)
df['area_per_room'] = df['Area_SqFt'] / df['total_rooms'].replace(0, 1)

# 3. Property age features
current_year = 2026
df['property_age'] = current_year - df['Built_Year_Clean']
df['property_age_squared'] = df['property_age'] ** 2  # Non-linear age effect
df['is_new'] = (df['Built_Year_Clean'] >= 2020).astype(int)  # Last 6 years
df['is_renovated'] = (df['Built_Year_Clean'] >= 2015).astype(int)

# 4. Luxury/amenity score
df['luxury_score'] = (
    df['furnished_encoded'] * 2 +
    df['servant_quarters'] * 2 +
    df['parking_spaces'] * 1 +
    df['total_rooms'] * 0.5
)

# 5. Density features
df['density'] = df['total_rooms'] / df['Area_SqFt'] * 1000
df['bed_density'] = df['Beds_Num'] / df['Area_SqFt'] * 1000

# 6. Interaction features
df['area_x_beds'] = df['Area_SqFt'] * df['Beds_Num']
df['area_x_floors'] = df['Area_SqFt'] * df['floors']
df['beds_x_baths'] = df['Beds_Num'] * df['Baths_Num']

# 7. Categorical feature encoding (already have property_type_encoded)
# Add more granular property types if available
df['is_house'] = (df['property_type_encoded'] == 0).astype(int)
df['is_flat'] = (df['property_type_encoded'] == 1).astype(int)
df['is_farm'] = (df['property_type_encoded'] == 2).astype(int)

# 8. Parking features
df['has_parking'] = (df['parking_spaces'] > 0).astype(int)
df['ample_parking'] = (df['parking_spaces'] >= 2).astype(int)

# 9. Floor features
df['is_multi_floor'] = (df['floors'] >= 2).astype(int)
df['floor_complexity'] = df['floors'] * df['total_rooms']

df['log_area'] = np.log1p(df['Area_SqFt'])
df['log_parking'] = np.log1p(df['parking_spaces'])



# Keep original + engineered features
feature_columns = [
    # Original features
    'Area_SqFt', 'Beds_Num', 'Baths_Num', 'Built_Year_Clean',
    'servant_quarters', 'floors', 'parking_spaces',
    'furnished_encoded', 'property_type_encoded',

    # after feature enigneering
    'total_rooms', 'rooms_per_floor', 'bed_bath_ratio',
    'area_per_bedroom', 'area_per_room',
    'property_age', 'property_age_squared', 'is_new',
    'luxury_score', 'density',
    'area_x_beds', 'beds_x_baths',
    'is_house', 'has_parking', 'is_multi_floor',
    'log_area'
]

# Keep only existing columns
existing_features = [col for col in feature_columns if col in df.columns]
df_engineered = df[existing_features + ['Price_PKR']]

print(f"Original features: 9")
print(f"Engineered features: {len(existing_features)}")
print(f"Total features: {len(existing_features)}")



Original features: 9
Engineered features: 25
Total features: 25


## Linear Regression

In [152]:
model = LinearRegression()
model.fit(X_train_scaled, y_train)

y_pred = model.predict(X_test_scaled)

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)


print(f"MAE:  {mae:,.0f} PKR")
print(f"RMSE: {rmse:,.0f} PKR")
print(f"R²:   {r2:.4f}")

results.append({
    'Model': 'Linear Regression',
    'MAE': mean_absolute_error(y_test, y_pred),
    'RMSE': np.sqrt(mean_squared_error(y_test, y_pred)),
    'R2': r2_score(y_test, y_pred)
})

MAE:  7,378,254 PKR
RMSE: 11,909,015 PKR
R²:   0.3630


## Random Forest

In [153]:
# First, i used normal random forest and then added feature enigneering to improve it
X = df_engineered.drop(columns=['Price_PKR'])
y = df_engineered['Price_PKR']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

rf_eng = RandomForestRegressor(random_state=42, n_estimators=100)
rf_eng.fit(X_train, y_train)

y_pred = rf_eng.predict(X_test)
r2_new = r2_score(y_test, y_pred)

print(f"\n Random Forest with Feature Engineering:")
print(f"   R² Score: {r2_new:.4f}")

print(f"   Original R²: 0.7255")
print(f"   New R²:  {r2_new:.4f}")


 Random Forest with Feature Engineering:
   R² Score: 0.7435
   Original R²: 0.7255
   New R²:  0.7435


## Decision Tree

In [154]:
model = DecisionTreeRegressor(random_state=42)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print(f"MAE:  {mae:,.0f} PKR")
print(f"RMSE: {rmse:,.0f} PKR")
print(f"R²:   {r2:.4f}")

results.append({
    'Model': 'Decision Tree',
    'MAE': mean_absolute_error(y_test, y_pred),
    'RMSE': np.sqrt(mean_squared_error(y_test, y_pred)),
    'R2': r2_score(y_test, y_pred)
})

MAE:  5,970,833 PKR
RMSE: 8,972,121 PKR
R²:   0.6384


## Gradient Boosting

In [155]:
model = GradientBoostingClassifier(random_state=42)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)


print(f"MAE:  {mean_absolute_error(y_test, y_pred):,.0f} PKR")
print(f"RMSE: {np.sqrt(mean_squared_error(y_test, y_pred)):,.0f} PKR")
print(f"R²:   {r2_score(y_test, y_pred):.4f}")

results.append({
    'Model': 'Gradient Boosting',
    'MAE': mean_absolute_error(y_test, y_pred),
    'RMSE': np.sqrt(mean_squared_error(y_test, y_pred)),
    'R2': r2_score(y_test, y_pred)
})

MAE:  6,932,500 PKR
RMSE: 11,274,553 PKR
R²:   0.4290


## XGBoost

In [156]:
model = XGBRegressor(random_state=42, verbosity=0)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)


print(f"MAE:  {mean_absolute_error(y_test, y_pred):,.0f} PKR")
print(f"RMSE: {np.sqrt(mean_squared_error(y_test, y_pred)):,.0f} PKR")
print(f"R²:   {r2_score(y_test, y_pred):.4f}")

results.append({
    'Model': 'XGboost',
    'MAE': mean_absolute_error(y_test, y_pred),
    'RMSE': np.sqrt(mean_squared_error(y_test, y_pred)),
    'R2': r2_score(y_test, y_pred)
})


MAE:  5,998,624 PKR
RMSE: 8,647,719 PKR
R²:   0.6641


## Catboost

In [157]:
print(f"Normal CatBoost R²: 0.7442")

tuned_catboost = CatBoostRegressor( iterations=150,learning_rate=0.05,depth=6, l2_leaf_reg=5, border_count=64,bagging_temperature=1, random_seed=42, verbose=0)

tuned_catboost.fit(X_train, y_train)
y_pred = tuned_catboost.predict(X_test)
print(f"Hyperparameterized CatBoost R²: {r2_score(y_test, y_pred):.4f}")
print(f"MAE:  {mean_absolute_error(y_test, y_pred):,.0f} PKR")
print(f"RMSE: {np.sqrt(mean_squared_error(y_test, y_pred)):,.0f} PKR")
print(f"R²:   {r2_score(y_test, y_pred):.4f}")

results.append({
    'Model': 'CatBoost',
    'MAE': mean_absolute_error(y_test, y_pred),
    'RMSE': np.sqrt(mean_squared_error(y_test, y_pred)),
    'R2': r2_score(y_test, y_pred)
})


Hyperparameterized CatBoost R²: 0.7869
MAE:  5,120,364 PKR
RMSE: 6,887,629 PKR
R²:   0.7869


## Model Comparsion

In [158]:
results_df = pd.DataFrame(results)
results_df = results_df.sort_values('R2', ascending=False)

print("\n" + "=" * 60)
print("MODEL COMPARISON (Best to Worst)")
print("=" * 60)
print(results_df.to_string(index=False))
print("\n BEST MODEL:", results_df.iloc[0]['Model'])



MODEL COMPARISON (Best to Worst)
            Model          MAE         RMSE       R2
         CatBoost 5.120364e+06 6.887629e+06 0.786911
          XGboost 5.998624e+06 8.647719e+06 0.664089
    Decision Tree 5.970833e+06 8.972121e+06 0.638414
Gradient Boosting 6.932500e+06 1.127455e+07 0.429021
Linear Regression 7.378254e+06 1.190901e+07 0.362951

 BEST MODEL: CatBoost


## Prediction System

In [162]:
best_model =  CatBoostRegressor( iterations=150,learning_rate=0.05,depth=6, l2_leaf_reg=5, border_count=64,bagging_temperature=1, random_seed=42, verbose=0)
best_model.fit(X_train, y_train)

print("\nEnter property details to predict the price:\n")

# Get user input
property_type = input("Property Type (House/Flat/Farm House): ").strip()
area_sqft = float(input("Area (in Square Feet): "))
beds = int(input("Number of Bedrooms: "))
baths = int(input("Number of Bathrooms: "))
built_year = int(input("Year Built (e.g., 2024): "))
servant_quarters = int(input("Servant Quarters (0/1/2/3): "))
floors = int(input("Number of Floors: "))
parking = int(input("Parking Spaces: "))
furnished = input("Furnished (Yes/No): ").strip()

# Encode categorical inputs
if furnished.lower() == 'yes':
    furnished_encoded = 1
else:
    furnished_encoded = 0

if property_type.lower() == 'house':
    property_type_encoded = 0
elif property_type.lower() == 'flat':
    property_type_encoded = 1
elif property_type.lower() == 'farm house':
    property_type_encoded = 2
else:
    property_type_encoded = 0

# Calculate ALL engineered features (same as during training)
current_year = 2026

total_rooms = beds + baths
rooms_per_floor = total_rooms / max(floors, 1)
bed_bath_ratio = beds / max(baths, 1)
area_per_bedroom = area_sqft / max(beds, 1)
area_per_room = area_sqft / max(total_rooms, 1)
property_age = current_year - built_year
property_age_squared = property_age ** 2
is_new = 1 if built_year >= 2020 else 0
is_renovated = 1 if built_year >= 2015 else 0

luxury_score = (furnished_encoded * 2 +
                servant_quarters * 2 +
                parking * 1 +
                total_rooms * 0.5)

density = total_rooms / area_sqft * 1000
bed_density = beds / area_sqft * 1000
area_x_beds = area_sqft * beds
area_x_floors = area_sqft * floors
beds_x_baths = beds * baths

is_house = 1 if property_type_encoded == 0 else 0
is_flat = 1 if property_type_encoded == 1 else 0
is_farm = 1 if property_type_encoded == 2 else 0
has_parking = 1 if parking > 0 else 0
ample_parking = 1 if parking >= 2 else 0
is_multi_floor = 1 if floors >= 2 else 0
floor_complexity = floors * total_rooms
log_area = np.log1p(area_sqft)
log_parking = np.log1p(parking)

# Create input data with ALL 25 engineered features
input_data = pd.DataFrame([[
    area_sqft, beds, baths, built_year, servant_quarters,
    floors, parking, furnished_encoded, property_type_encoded,
    total_rooms, rooms_per_floor, bed_bath_ratio,
    area_per_bedroom, area_per_room, property_age,
    property_age_squared, is_new, luxury_score, density,
    area_x_beds, beds_x_baths, is_house, has_parking,
    is_multi_floor, log_area
]], columns=[
    'Area_SqFt', 'Beds_Num', 'Baths_Num', 'Built_Year_Clean',
    'servant_quarters', 'floors', 'parking_spaces',
    'furnished_encoded', 'property_type_encoded',
    'total_rooms', 'rooms_per_floor', 'bed_bath_ratio',
    'area_per_bedroom', 'area_per_room', 'property_age',
    'property_age_squared', 'is_new', 'luxury_score', 'density',
    'area_x_beds', 'beds_x_baths', 'is_house', 'has_parking',
    'is_multi_floor', 'log_area'
])

# Predict
predicted_price = best_model.predict(input_data)[0]


print(" PREDICTION RESULT: ")

print(f"\nProperty Details:")
print(f"  • Type: {property_type}")
print(f"  • Area: {area_sqft:,.0f} sq ft ({area_sqft/225:.1f} Marla)")
print(f"  • Bedrooms: {beds}")
print(f"  • Bathrooms: {baths}")
print(f"  • Year Built: {built_year}")
print(f"  • Servant Quarters: {servant_quarters}")
print(f"  • Floors: {floors}")
print(f"  • Parking: {parking}")
print(f"  • Furnished: {furnished}")

print(f"\n Predicted Price: PKR {predicted_price:,.0f}")
print(f"   (Approximately PKR {predicted_price/10000000:.2f} Crore)\n")

 PREDICTION RESULT: 

Property Details:
  • Type: House
  • Area: 2,000 sq ft (8.9 Marla)
  • Bedrooms: 4
  • Bathrooms: 6
  • Year Built: 2024
  • Servant Quarters: 0
  • Floors: 2
  • Parking: 4
  • Furnished: Yes

 Predicted Price: PKR 32,582,237
   (Approximately PKR 3.26 Crore)

